# Demo Phi-3 Mini trên Colab T4

Notebook ngắn gọn theo `KH.pdf`: host `microsoft/Phi-3-mini-4k-instruct` trên GPU T4, hỏi một câu đơn giản, sau đó chạy benchmark Spark QA. Model tải khoảng 7.64 GB; nếu Colab runtime chưa reset thì lần sau sẽ đọc lại từ cache.

In [ ]:
# Chạy cell này trước khi import transformers/torch.
# Trên Colab, không cần restart runtime nếu các thư viện đã sẵn sàng.
%pip install -q "transformers>=4.40" "accelerate>=0.29" "huggingface_hub>=0.23"

## Lấy dữ liệu và mã benchmark từ GitHub

Cell này clone hoặc cập nhật repo vào `/content/a-triple-of-lms`. Benchmark sẽ lấy dataset từ `data/` và code từ `benchmarks/` trong repo này. Nếu chạy lại cell nhiều lần, notebook sẽ `git pull` thay vì clone lại.

In [ ]:
from pathlib import Path
import subprocess
import sys

GITHUB_REPO_URL = "https://github.com/Hutaph/a-triple-of-lms.git"
PROJECT_DIR = Path("/content/a-triple-of-lms")

if PROJECT_DIR.exists() and (PROJECT_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", GITHUB_REPO_URL, str(PROJECT_DIR)], check=True)

DATA_PATH = PROJECT_DIR / "data" / "spark_interview_questions.json"
BENCHMARK_DIR = PROJECT_DIR / "benchmarks"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy dataset: {DATA_PATH}")
if not (BENCHMARK_DIR / "spark_qa_benchmark.py").exists():
    raise FileNotFoundError(f"Không tìm thấy benchmark script: {BENCHMARK_DIR / 'spark_qa_benchmark.py'}")

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("Project:", PROJECT_DIR)
print("Dataset:", DATA_PATH)
print("Benchmark:", BENCHMARK_DIR / "spark_qa_benchmark.py")

## Tải model về local runtime

Notebook không mount Google Drive. Model sẽ được tải vào local runtime của Colab, mặc định là `/content/models/phi3-mini-4k-instruct`, và cache Hugging Face nằm trong `/content/hf_cache`. Khi runtime bị reset thì các thư mục này mất, chỉ cần chạy lại cell clone GitHub và cell tải model.

In [ ]:
from pathlib import Path
import os

RUNTIME_ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
HF_CACHE_DIR = RUNTIME_ROOT / "hf_cache"
LOCAL_MODEL_DIR = RUNTIME_ROOT / "models" / "phi3-mini-4k-instruct"

HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE_DIR)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE_DIR / "hub")
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE_DIR / "transformers")

print("Runtime root:", RUNTIME_ROOT)
print("Hugging Face cache:", HF_CACHE_DIR)
print("Local model dir:", LOCAL_MODEL_DIR)

In [ ]:
import gc
import os
import time

# Chỉ cần cho Windows/Anaconda; trên Colab dòng này không ảnh hưởng.
if os.name == "nt":
    os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Không có GPU. Phi-3 mini vẫn có thể chạy CPU nhưng rất chậm.")

In [ ]:
# Load tokenizer + model. Cell này có thể chạy lại sau lỗi mà không cần restart runtime.
# Quan trọng: không dùng trust_remote_code cho Phi-3 trên Colab.
# Remote code cũ của Phi-3 dễ gây lỗi RoPE: KeyError 'type' hoặc Unknown RoPE scaling type.

if "model" in globals():
    del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model_path = snapshot_download(
    repo_id=MODEL_ID,
    local_dir=str(LOCAL_MODEL_DIR),
    cache_dir=str(HF_CACHE_DIR / "hub"),
)
print("Model files:", model_path)

tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)

load_kwargs = {}

if torch.cuda.is_available():
    load_kwargs.update({
        "device_map": "cuda",
        "torch_dtype": torch.float16,
    })
else:
    load_kwargs.update({
        "device_map": "cpu",
        "torch_dtype": torch.float32,
    })

model = AutoModelForCausalLM.from_pretrained(model_path, local_files_only=True, **load_kwargs)
model.eval()

print("Loaded:", MODEL_ID)
print("From:", model_path)
print("Device:", next(model.parameters()).device)

In [ ]:
def ask_phi3(question: str, max_new_tokens: int = 120) -> str:
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt")
    device = next(model.parameters()).device
    inputs = {key: value.to(device) for key, value in inputs.items()}

    start = time.perf_counter()
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    latency = time.perf_counter() - start

    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    print(f"Latency: {latency:.1f}s")
    return answer

In [ ]:
question = "Explain MapReduce in exactly 5 bullet points. Each bullet must contain no more than 15 words."

print("Question:")
print(question)
print("\nPhi-3 output:")
print(ask_phi3(question))

## Benchmark Spark QA dataset

Phần này dùng `data/spark_interview_questions.json` và logic trong `benchmarks/spark_qa_benchmark.py` để chấm Phi-3. Mặc định notebook chạy full 80 câu. Nếu chỉ muốn kiểm tra nhanh, đổi `BENCHMARK_LIMIT = 5`.

In [ ]:
# Kiểm tra lại repo đã được clone và đưa project vào Python path.
from pathlib import Path
import sys

if "PROJECT_DIR" not in globals():
    PROJECT_DIR = Path("/content/a-triple-of-lms")

DATA_PATH = PROJECT_DIR / "data" / "spark_interview_questions.json"
BENCHMARK_DIR = PROJECT_DIR / "benchmarks"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Không tìm thấy dataset. Hãy chạy cell clone GitHub ở phía trên trước. "
        f"Đường dẫn đang kiểm tra: {DATA_PATH}"
    )
if not (BENCHMARK_DIR / "spark_qa_benchmark.py").exists():
    raise FileNotFoundError(
        "Không tìm thấy benchmark script. Hãy chạy cell clone GitHub ở phía trên trước. "
        f"Đường dẫn đang kiểm tra: {BENCHMARK_DIR / 'spark_qa_benchmark.py'}"
    )

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("Project:", PROJECT_DIR)
print("Dataset:", DATA_PATH)
print("Benchmark:", BENCHMARK_DIR / "spark_qa_benchmark.py")

In [ ]:
import csv
import inspect
import json
from pathlib import Path

from benchmarks import spark_qa_benchmark as benchmark

build_prompt = benchmark.build_prompt
composite_score = benchmark.composite_score
generate_answer = benchmark.generate_answer
save_visualizations = benchmark.save_visualizations
summarize = benchmark.summarize
write_csv = benchmark.write_csv


def score_answer(model_answer, reference_answer, must_have_points=None):
    if len(inspect.signature(composite_score).parameters) >= 3:
        return composite_score(model_answer, reference_answer, must_have_points)
    return composite_score(model_answer, reference_answer)


def write_human_review_template(results, path):
    if hasattr(benchmark, "write_human_review_template"):
        return benchmark.write_human_review_template(results, path)

    fields = [
        "id", "question", "category", "difficulty", "model_answer",
        "must_have_points", "auto_score", "correctness_1_5",
        "completeness_1_5", "tradeoff_1_5", "production_relevance_1_5",
        "clarity_1_5", "human_notes",
    ]
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fields)
        writer.writeheader()
        for result in results:
            writer.writerow({
                "id": result["id"],
                "question": result["question"],
                "category": result["category"],
                "difficulty": result["difficulty"],
                "model_answer": result["model_answer"],
                "must_have_points": json.dumps(result.get("must_have_points", []), ensure_ascii=False),
                "auto_score": result["score"],
                "correctness_1_5": "",
                "completeness_1_5": "",
                "tradeoff_1_5": "",
                "production_relevance_1_5": "",
                "clarity_1_5": "",
                "human_notes": "",
            })


def write_low_score_report(results, path, bottom_n=10):
    if hasattr(benchmark, "write_low_score_report"):
        return benchmark.write_low_score_report(results, path, bottom_n=bottom_n)

    lowest = sorted(results, key=lambda row: row["score"])[:bottom_n]
    lines = [
        "# Low Score Analysis",
        "",
        "Các câu dưới đây có điểm tự động thấp nhất. Nên đọc thủ công để phân biệt lỗi thật của model với hạn chế của metric token/keyword.",
        "",
    ]
    for result in lowest:
        lines.extend([
            f"## {result['id']}. {result['question']}",
            "",
            f"- Category: `{result['category']}`",
            f"- Difficulty: `{result['difficulty']}`",
            f"- Auto score: `{result['score']}`",
            f"- Token F1: `{result.get('token_f1', '')}`",
            f"- ROUGE-L: `{result.get('rouge_l', '')}`",
            f"- Keyword coverage: `{result.get('keyword_coverage', '')}`",
            "",
            "Must-have points:",
        ])
        for point in result.get("must_have_points", []):
            lines.append(f"- {point}")
        lines.extend(["", "Model answer:", "", result["model_answer"], ""])

    path.write_text("\n".join(lines) + "\n", encoding="utf-8")

BENCHMARK_LIMIT = 0  # 0 = chạy full 80 câu; đổi thành 5 nếu chỉ muốn test nhanh.
MAX_NEW_TOKENS = 220
OUTPUT_DIR = PROJECT_DIR / "benchmark_results"

dataset = json.loads(DATA_PATH.read_text(encoding="utf-8"))
if BENCHMARK_LIMIT:
    dataset = dataset[:BENCHMARK_LIMIT]

print("Benchmark questions:", len(dataset))

In [ ]:
results = []

for index, item in enumerate(dataset, 1):
    prompt = build_prompt(item["question"])
    model_answer, latency_s, output_tokens = generate_answer(
        tokenizer=tokenizer,
        model=model,
        prompt=prompt,
        max_new_tokens=MAX_NEW_TOKENS,
    )
    scores = score_answer(
        model_answer,
        item["reference_answer"],
        item.get("must_have_points"),
    )
    result = {
        "id": index,
        "question": item["question"],
        "category": item["category"],
        "difficulty": item["difficulty"],
        "reference_answer": item["reference_answer"],
        "must_have_points": item.get("must_have_points", []),
        "model_answer": model_answer,
        "latency_s": round(latency_s, 3),
        "output_tokens": output_tokens,
        **scores,
    }
    results.append(result)
    print(
        f"[{index:03d}/{len(dataset):03d}] "
        f"{item['difficulty']} {item['category']} "
        f"score={result['score']:.4f} latency={result['latency_s']:.1f}s"
    )

summary = summarize(results)
summary["overall"]

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

details_path = OUTPUT_DIR / "spark_qa_benchmark_results.json"
summary_path = OUTPUT_DIR / "spark_qa_benchmark_summary.json"
csv_path = OUTPUT_DIR / "spark_qa_benchmark_results.csv"
human_review_path = OUTPUT_DIR / "human_review_template.csv"
low_score_report_path = OUTPUT_DIR / "low_score_analysis.md"

details_path.write_text(json.dumps(results, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
write_csv(results, csv_path)
write_human_review_template(results, human_review_path)
write_low_score_report(results, low_score_report_path)
chart_paths = save_visualizations(results, summary, OUTPUT_DIR)

print("Wrote:", details_path)
print("Wrote:", summary_path)
print("Wrote:", csv_path)
print("Wrote:", human_review_path)
print("Wrote:", low_score_report_path)

In [ ]:
# Hiển thị charts trong notebook.
from IPython.display import Image, display

for chart_path in chart_paths:
    print(chart_path)
    display(Image(filename=str(chart_path)))

In [ ]:
# Hiển thị báo cáo các câu có điểm thấp nhất.
from IPython.display import Markdown, display

display(Markdown(low_score_report_path.read_text(encoding="utf-8")))

## Nhận xét kết quả benchmark

Kết quả trong các biểu đồ cần được đọc cùng giá trị `n` trên từng nhóm. Nếu `BENCHMARK_LIMIT = 0`, notebook đang chạy toàn bộ 80 câu và kết quả có tính đại diện tốt hơn. Nếu đổi về `BENCHMARK_LIMIT = 5`, biểu đồ chỉ là smoke test để kiểm tra pipeline.

**Theo độ khó**, biểu đồ cho biết model giữ chất lượng ổn định hay giảm điểm khi chuyển từ `easy` sang `medium` và `hard`. Nếu nhóm nào có `n` quá thấp, không nên kết luận mạnh từ nhóm đó.

**Theo category**, nhóm có điểm thấp là nơi nên đọc thủ công các câu trả lời. Với Spark, các category dễ làm model mất điểm thường là join optimization, skew, streaming state và fault tolerance, vì câu trả lời cần nhiều điều kiện sử dụng và trade-off production.

**Về latency**, thời gian trả lời trung bình khoảng 8 giây/câu trên T4 và khá ổn định giữa `medium` và `hard`. Với demo seminar, mức này chấp nhận được; nếu chạy full 80 câu thì tổng thời gian có thể vào khoảng 10-15 phút tùy độ dài output và tải GPU.

**Score vs latency** chưa cho thấy quan hệ rõ ràng giữa câu trả lời chậm hơn và điểm cao hơn. Một số câu có latency tương tự nhưng điểm khác nhau đáng kể, nghĩa là chất lượng phụ thuộc nhiều vào mức độ khớp ý với reference answer hơn là thời gian sinh.

## Các cải thiện đã thêm vào benchmark

Benchmark hiện tại không chỉ dùng `token_f1`, `rouge_l` và `keyword_coverage` nữa. Dataset đã có thêm `must_have_points`, và script chấm thêm `must_have_point_coverage` để đo model có nhắc đủ các ý chính bắt buộc hay không. Script cũng chấm `instruction_following` để kiểm tra độ dài, mức liên quan đến Spark, việc tránh refusal và việc có nhắc đến trade-off/production consideration.

Script cũng xuất thêm hai artifact phục vụ review thủ công: `human_review_template.csv` và `low_score_analysis.md`. File CSV cho phép chấm theo rubric 1-5 gồm correctness, completeness, trade-off, production relevance và clarity. File markdown liệt kê các câu điểm thấp nhất để phân tích lỗi định tính.

Các điểm nên làm tiếp nếu muốn benchmark nghiêm túc hơn:

1. Review thủ công 10-20 câu điểm thấp bằng `human_review_template.csv`.
2. So sánh nhiều model hoặc nhiều prompt trên cùng dataset, ví dụ Phi-3 vs Qwen hoặc prompt ngắn vs prompt có rubric.
3. Nếu dùng LLM-as-judge, nên dùng một model mạnh hơn model được chấm và giữ rubric cố định để giảm thiên lệch.

Kết luận ngắn: benchmark hiện tại đã đủ tốt để làm baseline tự động và có artifact phục vụ human review. Tuy nhiên, kết luận cuối cùng vẫn nên dựa trên cả điểm tự động lẫn phân tích thủ công các câu điểm thấp.

In [ ]:
# Xem nhanh câu điểm cao/thấp nhất để đưa vào slide demo.
ranked = sorted(results, key=lambda row: row["score"])

print("Lowest score:")
print(json.dumps(ranked[0], ensure_ascii=False, indent=2)[:2000])

print("\nHighest score:")
print(json.dumps(ranked[-1], ensure_ascii=False, indent=2)[:2000])